# 一階刷題：劇情背景

劇情（`data/stories/`）用到的背景圖。流程：**每個背景生成一批 → 挑一張 → 輸出 webp**。

- 模型和立繪一樣用 **Animagine XL 4.0 Opt**，參數也相同（CFG 5、28 steps、Euler a），畫風才會統一。
- 執行前：「執行階段 → 變更執行階段類型」選 GPU（有 L4 或 A100 就選，T4 也能跑但比較慢）。
- 全部用 `diffusers` 在 notebook 裡直接生成，不開 WebUI，符合 Colab 的使用規定。
- 每張圖的 seed、prompt 都存進 Google Drive 的 `nmle-art/`（和立繪同一個資料夾、同一份 `log.jsonl`），之後可以重現。
- 輸出的檔案放進 repo 的 `data/backgrounds/<背景 id>.webp`，網站就會自動換掉漸層色。

由上往下執行。需要你決定的地方都標了 **✏️**。

第 1 格裝完套件會自動重啟工作階段（畫面顯示工作階段當機是正常的），接著從第 2 格往下執行。

In [ ]:
# 1. 安裝套件
# 新的執行階段第一次跑時會安裝套件，裝完自動重啟工作階段（畫面會顯示工作階段當機，這是正常的），
# 接著從第 2 格繼續。同一個執行階段裡再跑這格會直接跳過。
import os
if not os.path.exists('/content/.nmle_bg_ready'):
    !pip install -q -U diffusers transformers accelerate safetensors
    # Colab 內建的 torchao 比新版 diffusers 需要的舊，裝著會讓 diffusers 載入失敗；我們用不到它
    !pip uninstall -y -q torchao
    open('/content/.nmle_bg_ready', 'w').close()
    print('安裝完成，正在重新啟動工作階段……請從第 2 格繼續執行')
    os.kill(os.getpid(), 9)
else:
    print('套件已經裝好，請執行第 2 格')

In [ ]:
# 2. GPU 與 Google Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 背景與模型設定
import gc, json, time
from pathlib import Path

import torch
from PIL import Image, ImageDraw
from IPython.display import display
from huggingface_hub import hf_hub_download
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

OUT = Path('/content/drive/MyDrive/nmle-art')
OUT.mkdir(parents=True, exist_ok=True)
WIDTH, HEIGHT = 1216, 832  # 橫式，SDXL 支援的解析度；網站上用 object-fit: cover 填滿畫面

# 背景 id 要和 data/stories/*.json 的 background 一致。
# 舞台是台灣南部的醫學大學，避免日式街景、日文招牌：負向 tag 有 japanese text。
# 祭典的劇情（溫芷薰「心動」）寫了鳥居和浴衣，所以 festival 保留日式元素。
BACKGROUNDS = {
    'cafe': {  # 自習咖啡廳「夜燈」：讀書會的固定地點，出現最多次
        'tags': 'cafe, indoors, wooden table, chair, window, night, warm lighting, ceiling light, '
                'potted plant, bookshelf, cup, cozy',
    },
    'library': {  # 圖書館：讀書、閉館前的書架間
        'tags': 'library, indoors, bookshelf, book, table, chair, window, sunlight, day, quiet',
    },
    'anatomy_lab': {  # 解剖教室：不要畫出大體或骨骼，只要空的教室與不鏽鋼桌
        'tags': 'laboratory, indoors, classroom, stainless steel table, cabinet, whiteboard, window, '
                'fluorescent light, tile floor, evening',
        'negative': 'skeleton, corpse, blood, gore',
    },
    'lab': {  # 微免實驗室：培養皿、顯微鏡
        'tags': 'laboratory, indoors, microscope, test tube, flask, petri dish, shelf, computer, window, '
                'counter, fluorescent light',
        'negative': 'blood, gore',
    },
    'hospital': {  # 醫院：病理科閱片室、藥局休息室都用這張，所以畫走廊與窗戶就好
        'tags': 'hospital, indoors, hallway, window, bench, fluorescent light, tile floor, door, evening',
        'negative': 'blood, gore, bed',
    },
    'track': {  # 操場跑道：傍晚
        'tags': 'running track, stadium, outdoors, field, bleachers, sky, cloud, sunset, orange sky',
    },
    'night_street': {  # 夜晚街道、宿舍前的路口
        'tags': 'street, outdoors, night, street lamp, road, sidewalk, building, power lines, night sky, '
                'scooter',
    },
    'festival': {  # 夏日祭典
        'tags': 'summer festival, outdoors, night, paper lantern, torii, festival stall, fireworks, night sky',
    },
    'stairwell': {  # 醫院逃生梯
        'tags': 'stairwell, stairs, indoors, handrail, window, fluorescent light, concrete, dim lighting',
        'negative': 'blood, gore',
    },
}
COMMON = 'no humans, scenery, detailed background, visual novel background'

SAFETY_NEG = 'nsfw, nude, explicit'
# 和立繪的負向 tag 不同：拿掉 scenery（這次要的就是場景），加上人物相關的 tag
NEGATIVE = ('1girl, 1boy, person, people, character, lowres, text, error, cropped, worst quality, low quality, '
            'low score, bad score, average score, signature, watermark, username, blurry, japanese text, '
            + SAFETY_NEG)
MODEL = {  # 和 portraits.ipynb 的 animagine 相同
    'repo': 'cagliostrolab/animagine-xl-4.0', 'file': 'animagine-xl-4.0-opt.safetensors',
    'cfg': 5.0, 'steps': 28, 'suffix': ', safe, masterpiece, high score, great score, absurdres',
}

# 重跑這格時先把上次載入的模型從 GPU 清掉，避免兩份 SDXL 同時佔著記憶體
if 'pipe' in globals() and pipe is not None:
    pipe = None
    gc.collect()
    torch.cuda.empty_cache()
pipe = None


def load_model():
    """載入模型（第一次會下載約 7 GB，要幾分鐘）。"""
    global pipe
    if pipe is not None:
        return
    path = hf_hub_download(MODEL['repo'], MODEL['file'])
    pipe = StableDiffusionXLPipeline.from_single_file(path, torch_dtype=torch.float16)
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe.to('cuda')
    print('已載入 Animagine XL 4.0 Opt')


def check_tokens(text, label):
    """SDXL 的文字編碼器只讀前 75 個 token，超過的會被默默丟掉。"""
    count = len(pipe.tokenizer(text).input_ids) - 2
    if count > 75:
        raise ValueError(f'{label} 有 {count} 個 token，超過 75，後面的 tag 會被丟掉。請刪減 tag。\n{text}')
    return count


def prompts(bg):
    b = BACKGROUNDS[bg]
    prompt = b['tags'] + ', ' + COMMON + MODEL['suffix']
    negative = NEGATIVE + (', ' + b['negative'] if b.get('negative') else '')
    check_tokens(prompt, f'{bg} 的 prompt')
    check_tokens(negative, f'{bg} 的負向 prompt')
    return prompt, negative


def log(record):
    with open(OUT / 'log.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


def generate(bg, seed):
    """產生一張背景，存到 Drive。同一個背景、同一個 seed 會得到同一張圖；已經生成過的直接讀檔。"""
    path = OUT / 'backgrounds' / bg / f'{seed}.png'
    if path.exists():
        return Image.open(path).convert('RGB')
    load_model()
    prompt, negative = prompts(bg)
    image = pipe(prompt=prompt, negative_prompt=negative, width=WIDTH, height=HEIGHT,
                 guidance_scale=MODEL['cfg'], num_inference_steps=MODEL['steps'],
                 generator=torch.Generator('cuda').manual_seed(seed)).images[0]
    path.parent.mkdir(parents=True, exist_ok=True)
    image.save(path)
    log({'kind': 'background', 'model': 'animagine', 'background': bg, 'seed': seed, 'prompt': prompt,
         'negative': negative, 'cfg': MODEL['cfg'], 'steps': MODEL['steps'], 'path': str(path),
         'time': time.time()})
    return image


def contact_sheet(images, labels, cols=3, thumb_w=400):
    """縮圖排成一張，方便比較。"""
    thumbs = [im.resize((thumb_w, round(im.height * thumb_w / im.width))) for im in images]
    h = max(t.height for t in thumbs)
    rows = (len(thumbs) + cols - 1) // cols
    sheet = Image.new('RGB', (thumb_w * cols, (h + 28) * rows), 'white')
    draw = ImageDraw.Draw(sheet)
    for i, (t, label) in enumerate(zip(thumbs, labels)):
        x, y = (i % cols) * thumb_w, (i // cols) * (h + 28)
        sheet.paste(t, (x, y + 28))
        draw.text((x + 6, y + 6), str(label), fill='black')
    return sheet


print('設定完成')

## 步驟 1：每個背景生成一批

先做劇情最常出現的五個：`cafe`、`library`、`anatomy_lab`、`lab`、`hospital`。

挑的時候看：

- **沒有人**：背景上有人物，立繪站上去會很怪。有的話換一張，或在那個背景的 `negative` 加 tag。
- **中間不要太花**：立繪會站在畫面正中間、對話框蓋住下方三分之一，重要的東西最好在兩側或上半部。
- **手機上只看得到中間**：手機是直的，橫式背景會裁掉左右兩邊。
- **沒有亂碼文字、招牌**：AI 畫的字通常是亂碼。
- **不要血腥**：解剖教室、醫院不要畫出大體、骨骼或血跡（尺度是輔導級）。

已經生成過的圖會直接從 Drive 讀，不會重畫；中途斷線重開，從第 1 格跑到這裡就能接著做。

In [ ]:
TODO = ['cafe', 'library', 'anatomy_lab', 'lab', 'hospital']  # ✏️ 之後再加 'track'、'night_street'、'festival'、'stairwell'
SEEDS = list(range(100, 109))  # ✏️ 每個背景 9 張；都不滿意就換一段數字，例如 range(200, 209)

for bg in TODO:
    images = [generate(bg, s) for s in SEEDS]
    print(f'=== {bg}')
    display(contact_sheet(images, [f'{bg} seed {s}' for s in SEEDS]))

## 步驟 2：輸出

每個背景各挑一張填進 `PICKS`。輸出成 `<背景 id>.webp` 並打包成 zip 下載；解壓後把裡面的檔案放進 repo 的 `data/backgrounds/`。

定稿後把選的 seed 記進 repo 的 `characters.md`「劇情背景」的定稿紀錄。

In [ ]:
PICKS = {  # ✏️ 每個背景選的 seed；還沒做的不用寫
    # 'cafe': 100,
    # 'library': 100,
}
assert PICKS, '先在 PICKS 填上每個背景選的 seed'

import shutil
from google.colab import files

export_dir = Path('/content/export/backgrounds')
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)
for bg, seed in PICKS.items():
    image = generate(bg, seed)
    image.save(export_dir / f'{bg}.webp', quality=85, method=6)
    print(f'{bg}：seed {seed}，{(export_dir / f"{bg}.webp").stat().st_size // 1024} KB')

display(contact_sheet([Image.open(export_dir / f'{bg}.webp') for bg in PICKS], list(PICKS)))
zip_path = shutil.make_archive('/content/backgrounds', 'zip', export_dir)
files.download(zip_path)